# LLM-as-a-judge

In [ ]:
import json
from json_repair import repair_json
import pandas as pd
import re
import os
import random
from IPython.display import clear_output, display
import time

import anthropic
from openai import OpenAI
from google import genai
from google.genai import types

from ZeroShotRecommendations import get_syllabus_data

In [ ]:
df = pd.read_csv("data/comments_df_subsample.csv")

df_syllabus = pd.read_csv("data/courses_sample_syllabus.csv")

with open('data/labeled - pratices and quotes all courses(50+1)_v2.json', "r") as f:
    labeled_comments = json.load(f) 

# with open('output/recommendations_raw+Classified-CollabirativeFiltering-v3.json', "r") as f:
#     recommendations = json.load(f) 

with open('output/recommendations_autonomousAgent2_v2.json', "r") as f:
    recommendations = json.load(f) 

# with open('output/recommendations_raw+Classified.json', "r") as f:
#     recommendations = json.load(f) 

In [ ]:
print(len(recommendations))

In [ ]:
recom = recommendations[0]
output = json.loads(recom['recommendation'])
# output = json.loads(recom['recommendation'])
output['used_tools']

In [ ]:
# recommendations[0]
# print(f"Base material for recommendation {recommendations[0]['base']} reviews")
print(f"Inference method: {recommendations[0]['method']}")
# print(f"Suplementary information: {recommendations[0]['supinf']}")
print(f"Recommendation:\n\n{recommendations[0]['recommendation'][0]}")
# print(recommendations[0]['recommendation'][:1000])

In [ ]:
rcms = []

# for recom in recommendations:
#     d = {'course_id' : recom['course_id']['course_id'],
#          'teaching' : recom['course_id']['teaching'],
#          'base' : recom['base'],
#          'supinf' : recom['supinf'],
#          'method' : recom['method'],
#          'recommendation' : recom['recommendation']
        
#     }
#     rcms.append(d)

# For autonomous agent:
for i, recom in enumerate(recommendations):
    try:
        # output = json.loads(repair_json(recom['recommendation']))
        # try:
        rx = recom['recommendation'][0]
        rx = rx.split("</think>")
        
        d = {'course_id' : recom['course_id'],
             # 'teaching' : recom['course_id']['teaching'],
             'base' : recom['base'],
             # 'tools' : output['used_tools'],
             'supinf' : recom['supinf'],
             'method' : recom['method'],
             'recommendation' : rx[0]# output['recommendation']
        }

        rcms.append(d)
        
    except:
        print(recom['recommendation'])

## Core Function and call clients

In [ ]:
def message_compose(course_id, objective, syllabi, comments, teaching_analysis, recommendation):
    
    schema = [
    {"item" : "The recommendation aligns with the course syllabus",
     "analysis of recommendation" : "<your analysis of the recommendation in relationship with the item>",
     "score" : "<1-7>"
    },
    {"item" : "The recommendation helps develop and adjust teaching strategies.",
     "analysis of recommendation" : "<your analysis of the recommendation in relationship with the item>",
     "score" : "<1-7>"
    },
    {"item" : "The recommendation is easily understandable.",
     "analysis of recommendation" : "<your analysis of the recommendation in relationship with the item>",
     "score" : "<1-7>"
    },
    {"item" : "The recommendation shows an understanding of the instructor's strengths and weaknesses.",
     "analysis of recommendation" : "<your analysis of the recommendation in relationship with the item>",
     "score" : "<1-7>"
    },
    {"item" : "The teaching strategies suggested by the recommendation seem feasible to carry out by the instructor of this course.",
     "analysis of recommendation" : "<your analysis of the recommendation in relationship with the item>",
     "score" : "<1-7>"
    }]
    
    random.shuffle(schema)


    prompt_ev = f"""You are an expert on teaching and learning in higher education. Your task is to assess the quality of teaching recommendations generated by an AI model.
    
## Context

### Course: {course_id}: 
{objective}

This is the course syllabi:
{syllabi}

This are student comments written in respective teaching survey of the course:
{comments}

This is a report with the analysis of teaching in this course, with detected teaching practices and verbatim quotes of student's reviews:
{teaching_analysis}

And this was the generated recommendations:
{recommendation}

## Task
Assess the quality of teaching recommendations generated by an AI model. Evaluate the given recommendation against several criteria using the following scale:
1 = Strongly Disagree
7 = Strongly Agree

## Output Requirements
Respond ONLY with a valid, raw JSON array. 
- Do NOT wrap the output in markdown code blocks (do not use ``` or `json`). 
- Do NOT include any conversational preamble, explanations, or summaries outside of the JSON.

You must strictly adhere to the following schema, using valid double quotes for all keys and string values:
{str(schema)}
Remember to return ONLY the json, without any other comment or summary out of it.
Do not add marks ```"""

    return prompt_ev

In [ ]:
Claude = anthropic.Anthropic(api_key = "your api key")

Gemini = genai.Client(api_key = "your api key")

Gpt = OpenAI(api_key="your api key")

## Compose messages

In [ ]:
eval_messages = []

for recom in rcms:
    crs = recom['course_id']
    
    objective, syllabi = get_syllabus_data(df_syllabus, crs)
    
    comments = df[df['course_id'] == crs]['reviews'].tolist()

    for l in labeled_comments:
        if l['course_id'] == crs:
            teaching_analysis  = l['teaching']

    recommendation  = recom['recommendation']
    
    msg = message_compose(course_id = crs, 
                    objective = objective, 
                    syllabi = syllabi, 
                    comments = comments, 
                    teaching_analysis = teaching_analysis, 
                    recommendation = recommendation)

    eval_messages.append(msg)

## Enviar batch a Claude

In [ ]:
reqs = []
req_info = []
req_id = 1

for recom in rcms:
    crs = recom['course_id']
    
    objective, syllabi = get_syllabus_data(df_syllabus, crs)
    
    comments = df[df['course_id'] == crs]['reviews'].tolist()
    
    # teaching_analysis  = recom['teaching'] # For static agents

    for l in labeled_comments:       # For autonomous Agent
        if l['course_id'] == crs:
            teaching_analysis  = l['teaching']

    recommendation  = recom['recommendation']    
          
    d = {"req_id" : f"request-{req_id}", # For static agents
         'course_id' : recom['course_id'],
         'base' : recom['base'],
         'supinf' : recom['supinf'],
         'method' : recom['method'],
        }

    # d = {"req_id" : f"request-{req_id}", # For autonomous agent
    #      'course_id' : recom['course_id'],
    #      'base' : "Raw",
    #      'tools' : recom['tools'],
    #      'method' : recom['method'],
    #     }
                
    request={
            "custom_id": f"request-{req_id}",
            "params": {
                "model": 'claude-sonnet-5',
                "max_tokens": 10000,
                "messages": [
                    {
                        "role": "user",
                        "content": message_compose(course_id = crs, 
                                                   objective = objective, 
                                                   syllabi = syllabi, 
                                                   comments = comments, 
                                                   teaching_analysis = teaching_analysis, 
                                                   recommendation = recommendation),
                    }
                ]
            },
        }
    reqs.append(request)

    req_info.append(d)
    req_id += 1

In [ ]:
message_batch = Claude.messages.batches.create(requests = reqs)

In [ ]:
print(message_batch.to_json())

In [ ]:
result = Claude.messages.batches.results(message_batch.id)

In [ ]:
Claude_output = []
for entry in result:
    Claude_output.append(entry.model_dump())

In [ ]:
with open("output/evaluations_output/claude_eval.json", "r") as f:
    Claude_output = json.load(f)

In [ ]:
claude_assessment = pd.DataFrame(columns = ['req_id', 'course_id', "base", "sup_inf", "method",
                                            "item",
                                            "analysis of recommendation",
                                            "score"])


for output in Claude_output:
    custom_id = output['custom_id']
    for rq in req_info:
        if rq['req_id'] == custom_id:
            info = rq

    if 'message' in output['result'].keys():
        
        try:
            text = output['result']['message']['content'][0]['text']
            text = text.replace("```json\n", "")
            text = text.replace("\n```", "")
            
            js = json.loads(repair_json(text))
    
    
            djs = pd.DataFrame(js)
    
            djs["req_id"] = info['req_id']
            djs["course_id"] = info['course_id']
            djs["base"] = info['base']
            djs["sup_inf"] = info['supinf']
            djs["method"] = info['method']
            
            claude_assessment = pd.concat([claude_assessment, djs], ignore_index=True)
            # claude_assessment.append(js)
        except:
            text = output['result']['message']['content'][1]['text']
            text = text.replace("```json\n", "")
            text = text.replace("\n```", "")
            
            js = json.loads(repair_json(text))
    
    
            djs = pd.DataFrame(js)
    
            djs["req_id"] = info['req_id']
            djs["course_id"] = info['course_id']
            djs["base"] = info['base']
            djs["sup_inf"] = info['supinf']
            djs["method"] = info['method']
            
            claude_assessment = pd.concat([claude_assessment, djs], ignore_index=True)
    else:
        print(output['result'].keys())

claude_assessment

## Gpt

In [ ]:
reqs = []
req_info = []
req_id = 1

for recom in rcms:
    crs = recom['course_id']
    
    objective, syllabi = get_syllabus_data(df_syllabus, crs)
    
    comments = df[df['course_id'] == crs]['reviews'].tolist()
    
    # for l in labeled_comments:       # For autonomous Agent
    #     if l['course_id'] == crs:
    #         teaching_analysis  = l['teaching']

    recommendation  = recom['recommendation']    
          
    d = {"req_id" : f"request-{req_id}", # For static agents
         'course_id' : recom['course_id'],
         'base' : recom['base'],
         'supinf' : recom['supinf'],
         'method' : recom['method'],
        }

    # d = {"req_id" : f"request-{req_id}", # For autonomous agent
    #      'course_id' : recom['course_id'],
    #      'base' : "Raw",
    #      'tools' : str(recom['tools']),
    #      'method' : recom['method'],
    #     }
                
    request = {
            "custom_id": f"request-{req_id}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-5.6-terra",
                "messages": [
                    {"role": "user", 
                     "content": message_compose(course_id = crs, 
                                                   objective = objective, 
                                                   syllabi = syllabi, 
                                                   comments = comments, 
                                                   teaching_analysis = teaching_analysis, 
                                                   recommendation = recommendation)}
                ],
                "temperature": 1,
            }
        }
    reqs.append(request)

    req_info.append(d)
    req_id += 1

In [ ]:
file_name = "gpt_input_batch.jsonl"

with open(file_name, 'w', encoding='utf-8') as f:
    for rq in reqs:
        f.write(json.dumps(rq, ensure_ascii=False) + '\n')

In [ ]:
with open(file_name, "rb") as file_stream:
    batch_input_file = Gpt.files.create(
        file=file_stream,
        purpose="batch"
    )

In [ ]:
batch_job = Gpt.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

In [ ]:
current_job = Gpt.batches.retrieve(batch_job.id)
status = current_job.status

print(f"State: {status}")

In [ ]:
Gptval = Gpt.files.content(current_job.output_file_id)

In [ ]:
gptext = Gptval.text
gptext = gptext.split("\n")

In [ ]:
lines = []

for line in gptext:
    line_js = json.loads(line)
    lines.append(line_js)
    

In [ ]:
with open("output/evaluations_output/gpt_eval.json", "w", encoding="utf-8") as f:
    json.dump(lines, f, ensure_ascii=False, indent=4, default=str)

## Gemini

### Compose messages

In [ ]:
bs = 0
reqsG = []
req_infoG = []

In [ ]:
req_id = 1
strt = 0
end = 599
splts = [599, 1199]
req_id += strt


finisheds = ["JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_PARTIALLY_SUCCEEDED",
            "JOB_STATE_EXPIRED" ]

Gemini_assessments = pd.DataFrame(columns = ['course_id', "base", "sup_inf", "method",
                                            "item",
                                            "analysis of recommendation",
                                            "score"])

generation_config = {
    'maxOutputTokens': 10000,
    'topP': 0.95
}

sub_batch = []

for recom in rcms[strt:end]:
    
    crs = recom['course_id']
    
    objective, syllabi = get_syllabus_data(df_syllabus, crs)
    
    comments = df[df['course_id'] == crs]['reviews'].tolist()
    
    teaching_analysis  = recom['teaching']

    recommendation  = recom['recommendation']    
          
    d = {"req_id" : f"request-{req_id}",
         'course_id' : recom['course_id'],
         'base' : recom['base'],
         'supinf' : recom['supinf'],
         'method' : recom['method'],
        }
    
    input_txt = message_compose(course_id = crs, 
                    objective = objective, 
                    syllabi = syllabi, 
                    comments = comments, 
                    teaching_analysis = teaching_analysis, 
                    recommendation = recommendation)
    
    req = {
        "id": f"request-{req_id}",
        "request": {'contents': [{
            'parts': [{'text': input_txt}],
            'role': 'user'
        }],
            "generationConfig": generation_config 
        }
    }

    reqsG.append(req)
    sub_batch.append(req)
    req_infoG.append(d)
    req_id += 1
    
if req_id in splts:
    filename = f"Gemini_inputBatch_{req_id}.jsonl"

    with open(filename, "w") as f:
        for req in sub_batch:
            f.write(json.dumps(req) + "\n")
            
    print(f"Written: {filename}")
    # sub_batch = []

        # filename = f"Gemini_inputBatch_{req_id}.jsonl"
        
        # with open(filename, "w") as f:
        #     for req in sub_batch:
        #         f.write(json.dumps(req) + "\n")
                
        # subir batch
uploaded_file = Gemini.files.upload(
    file=filename,
    config=types.UploadFileConfig(display_name=filename, mime_type='jsonl')
)
        
        # print(f"Uploaded file: {uploaded_file.name}")


file_batch_job = Gemini.batches.create(
    model = 'gemini-3.1-pro-preview',
    src = uploaded_file.name,
    config={
        'display_name': f"file-upload-job-{req_id}",
        },
    )
print(f"State: {uploaded_file.state}")
        
stt = Gemini.batches.get(name=file_batch_job.name).state
    
while stt not in finisheds:
    time.sleep(60)
    stt = Gemini.batches.get(name=file_batch_job.name).state
    print(f"State: {stt}")
        
if stt == "JOB_STATE_SUCCEEDED":
    fbj = Gemini.batches.get(name=file_batch_job.name)
    output_file_name = fbj.dest.file_name
    print(f"Downloaded {output_file_name}")
    output_content = Gemini.files.download(file=output_file_name)
    
    text_content = output_content.decode('utf-8')
    print(f"Retrieved and decoded {str(text_content)[:300]}")

    Gemini_json = []
    for line in text_content.strip().split('\n'):
        if line.strip():
            data = json.loads(line)
            Gemini_json.append(data)
            
            filename = f"Gemini_outputBatch_{req_id}.json"
            
            with open(filename, "w") as f:
                json.dump(Gemini_json, f, indent = 2)
            print(f"Written: {filename}\nContinuing")
            sub_batch = []
            bs += 1
            
        time.sleep(60)

print(f"Finished. Processed {len(reqsG)} reqs in {bs} batches.")

In [ ]:
stt = Gemini.batches.get(name=file_batch_job.name).state
    
while stt not in finisheds:
    time.sleep(60)
    stt = Gemini.batches.get(name=file_batch_job.name).state
    print(f"State: {stt}")

if stt == "JOB_STATE_SUCCEEDED":
    fbj = Gemini.batches.get(name=file_batch_job.name)
    output_file_name = fbj.dest.file_name
    print(f"Downloaded {output_file_name}")
    output_content = Gemini.files.download(file=output_file_name)

    text_content = output_content.decode('utf-8')
    print(f"Retrieved and decoded {str(text_content)[:300]}")
    # 2. Parsear el JSONL
    Gemini_json = []
    for line in text_content.strip().split('\n'):
        if line.strip(): # Evitar líneas vacías
            data = json.loads(line)
            Gemini_json.append(data)
    
    filename = f"Gemini_outputBatch_{600}.json"
    
    with open(filename, "w") as f:
        json.dump(Gemini_json, f, indent = 2)
    print(f"Written: {filename}\nContinuing")
    sub_batch = []
    bs += 1

In [ ]:
## For autonomous agent:
bs = 0
req_id = 0
reqsG = []
req_infoG = []

finisheds = ["JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_PARTIALLY_SUCCEEDED",
            "JOB_STATE_EXPIRED" ]

# Gemini_assessments = pd.DataFrame(columns = ['course_id', "base", "sup_inf", "method",
                                            # "item",
                                            # "analysis of recommendation",
                                            # "score"])

generation_config = {
    'maxOutputTokens': 10000,
    'topP': 0.95
}

sub_batch = []

for recom in rcms:
    
    crs = recom['course_id']
    
    objective, syllabi = get_syllabus_data(df_syllabus, crs)
    
    comments = df[df['course_id'] == crs]['reviews'].tolist()
    
    for l in labeled_comments:       # For autonomous Agent
        if l['course_id'] == crs:
            teaching_analysis  = l['teaching']

    recommendation  = recom['recommendation']    
          
    d = {"req_id" : f"request-{req_id}", # For static agents
         'course_id' : recom['course_id'],
         'base' : recom['base'],
         'supinf' : recom['supinf'],
         'method' : recom['method'],
        }

    # d = {"req_id" : f"request-{req_id}", # For autonomous agent
    #      'course_id' : recom['course_id'],
    #      'base' : "raw",
    #      'tools' : recom["tools"],
    #      'method' : recom['method'],
    #     }
    
    input_txt = message_compose(course_id = crs, 
                    objective = objective, 
                    syllabi = syllabi, 
                    comments = comments, 
                    teaching_analysis = teaching_analysis, 
                    recommendation = recommendation)
    
    req = {
        "id": f"request-{req_id}",
        "request": {'contents': [{
            'parts': [{'text': input_txt}],
            'role': 'user'
        }],
            "generationConfig": generation_config 
        }
    }

    reqsG.append(req)
    sub_batch.append(req)
    req_infoG.append(d)
    req_id += 1

    if req_id == 599:
        sub_batch = []
    

filename = f"Gemini_inputBatch_{req_id}.jsonl"

with open(filename, "w") as f:
    for req in sub_batch:
        f.write(json.dumps(req) + "\n")
        
print(f"Written: {filename}")

In [ ]:
len(sub_batch)

In [ ]:
filename = f"Gemini_inputBatch_{req_id}.jsonl"

with open(filename, "w") as f:
    for req in sub_batch:
        f.write(json.dumps(req) + "\n")
        
print(f"Written: {filename}")

uploaded_file = Gemini.files.upload(
    file=filename,
    config=types.UploadFileConfig(display_name=filename, mime_type='jsonl')
)

print(f"Uploaded file: {uploaded_file.name}")


file_batch_job = Gemini.batches.create(
    model = 'gemini-3.1-pro-preview',
    src = uploaded_file.name,
    config={
        'display_name': f"file-upload-job-{req_id}",
        },
    )
print(f"State: {uploaded_file.state}")

stt = Gemini.batches.get(name=file_batch_job.name).state



In [ ]:
finisheds = ["JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_PARTIALLY_SUCCEEDED",
            "JOB_STATE_EXPIRED" ]

while stt not in finisheds:
    stt = Gemini.batches.get(name=file_batch_job.name).state
    print(f"State: {stt}")
    time.sleep(60)

In [ ]:
fbj = Gemini.batches.get(name=file_batch_job.name)
output_file_name = fbj.dest.file_name
print(f"Downloaded {output_file_name}")
output_content = Gemini.files.download(file=output_file_name)

text_content = output_content.decode('utf-8')
print(f"Retrieved and decoded {str(text_content)[:300]}")

Gemini_json = []
for line in text_content.strip().split('\n'):
    if line.strip():
        data = json.loads(line)
        Gemini_json.append(data)

output_filename = f"Gemini_outputBatch_retrievedStrengths2.json"

with open(output_filename, "w") as f:
    json.dump(Gemini_json, f, indent = 2)

print(f"Writted {output_filename}")